In [1]:
import numpy as np
import pandas as pd
import fiftyone.zoo as foz

In [2]:
# 1. Download Open Images dataset and match images with labels
#dataset = foz.load_zoo_dataset(
    #"open-images-v7",
    #split="train",
    #label_types=["detections"],
    #classes=["Knife", "Scissors",  "Hammer", "Screwdriver","Baseball bat"],
    #max_samples=2000,
    #dataset_name="weapon_dataset")

In [3]:
# 2. Load the dataset 
import fiftyone as fo

dataset = fo.load_dataset("weapon_dataset")

In [4]:
# 3. Keep only the classes you want
wanted_classes = [
    "Knife",
    "Scissors",
    "Hammer",
    "Screwdriver",
    "Baseball bat"
]

dataset = dataset.filter_labels(
    "ground_truth",
    fo.ViewField("label").is_in(wanted_classes)
)

In [5]:
# 1600 for training
train_dataset = dataset.take(1600, seed=42)

# Remaining 400 for validation
val_dataset = dataset.exclude(train_dataset)


In [6]:
# 4. Export directly to YOLO format
from pathlib import Path

YOLO_DIR = Path(r"E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo")

train_dataset.export(
    export_dir=str(YOLO_DIR / "train"),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    classes=wanted_classes
)

val_dataset.export(
    export_dir=str(YOLO_DIR / "val"),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    classes=wanted_classes
)

Directory 'E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\train' already exists; export will be merged with existing files
 100% |███████████████| 1600/1600 [7.1s elapsed, 0s remaining, 230.7 samples/s]      
Directory 'E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\val' already exists; export will be merged with existing files
 100% |█████████████████| 400/400 [1.6s elapsed, 0s remaining, 244.6 samples/s]         


In [7]:
yaml = f"""path: {YOLO_DIR.as_posix()}
train: train/images
val: val/images
names: {wanted_classes}"""

(YOLO_DIR / "dataset.yaml").write_text(yaml)

167

In [8]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

In [10]:
# 5. Train the model # Disable Ultralytics logging

results = model.train(
    data=r"E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\dataset.yaml",
    epochs=10,
    imgsz=640,
    batch=30
)

print("Training completed!")

New https://pypi.org/project/ultralytics/8.4.123 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.65  Python-3.13.5 torch-2.12.0+cpu CPU (AMD Ryzen 5 7235HS)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=30, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosa

In [11]:
#  Validate the trained model
metrics = model.val()

Ultralytics 8.4.65  Python-3.13.5 torch-2.12.0+cpu CPU (AMD Ryzen 5 7235HS)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 609.9338.4 MB/s, size: 305.4 KB)
val: Scanning E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\val\labels\val.cache... 400 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 400/400 69.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 1.2s/it 29.9s1.3ss
                   all        400        512      0.784      0.511      0.565      0.403
                 Knife        113        175      0.697      0.644      0.699      0.507
              Scissors         63         91      0.833      0.713      0.824      0.631
                Hammer         28         31      0.584      0.516      0.425      0.334
           Screwdriver          9         11          1          0       0.14      0.113
          Ba

In [27]:
rom pathlib import Path

VAL_DIR = Path(r"E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\val\images\val")
images = sorted(VAL_DIR.glob("*.*"))

n = int(input("Enter image number: "))
image_path = images[n - 1]

results = model.predict(source=str(image_path), save=True)
results[0].show()

Enter image number:  99



image 1/1 E:\DS\Deep learning\Objcet Detection Proect\weapon_yolo\val\images\val\2082f1cfde105fa6.jpg: 448x640 1 Knife, 1 Scissors, 82.8ms
Speed: 4.9ms preprocess, 82.8ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
Results saved to E:\DS\Deep learning\Objcet Detection Proect\runs\detect\predict


In [31]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA available: False
GPU: No GPU
